In [22]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ipywidgets import VBox, HBox, FloatSlider, Output, FloatLogSlider, Button, Checkbox, Layout
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.interpolate import RegularGridInterpolator
import json

%matplotlib inline

In [23]:


def make_array_file_widgets(get_array, set_array, default_path="default_array.json"):
    """
    Create widgets for saving/loading a 2D Python array.

    Parameters
    ----------
    get_array : callable
        Function returning the current 2D array.
    set_array : callable
        Function accepting a new 2D array and updating your editor/display.
    default_path : str
        Default filename/path to save to or load from.
    """

    path_box = widgets.Text(
        value=default_path,
        description="File:",
        layout=widgets.Layout(width="400px")
    )

    save_button = widgets.Button(
        description="Save array",
        button_style="success",
        icon="save"
    )

    load_button = widgets.Button(
        description="Load array",
        button_style="info",
        icon="folder-open"
    )

    status = widgets.Output()

    def save_array(_):
        with status:
            status.clear_output()
            try:
                array = get_array()
                with open(path_box.value, "w", encoding="utf-8") as f:
                    json.dump(array, f, indent=2)
                print(f"Saved array to {path_box.value}")
            except Exception as e:
                print(f"Error saving array: {e}")

    def load_array(_):
        with status:
            status.clear_output()
            try:
                with open(path_box.value, "r", encoding="utf-8") as f:
                    array = json.load(f)

                set_array(array)
                print(f"Loaded array from {path_box.value}")
            except Exception as e:
                print(f"Error loading array: {e}")

    save_button.on_click(save_array)
    load_button.on_click(load_array)

    return widgets.VBox([
        widgets.HBox([path_box, save_button, load_button]),
        status
    ])

In [39]:
c = 344
n = 128
fig_width = 13
fov = np.pi / 2
todeg = 1 / np.pi * 180

polar_pts = np.meshgrid(
    np.linspace(-np.pi, np.pi, 7 * n),
    np.linspace(0, fov * todeg, n))
cart_pts = np.stack((
    np.sin(polar_pts[0]) * polar_pts[1] / todeg,
    np.cos(polar_pts[0]) * polar_pts[1] / todeg), axis = -1)

ffov = fov + 1e-5
x = np.linspace(-ffov, ffov, n)
y = np.linspace(-ffov, ffov, n)

plot_points = (polar_pts, cart_pts, (x, y))

w_f = FloatLogSlider(
    value=4000, base = 10, min=2, max=np.log10(6000),step=0.001,
    description="Frequency", continuous_update=True, readout_format=".1f"
)
w_idx = FloatSlider(
    value=0, min=0, max=16, step=1,
    description="Index", continuous_update=True, readout_format=".0f"
)
w_y = FloatSlider(
    value=0, min=0, max=0.32, step=0.001,
    description="Y", continuous_update=True, readout_format=".3f"
)
w_x = FloatSlider(
    value=0, min=0, max=0.277, step=0.001,
    description="X", continuous_update=True, readout_format=".3f"
)
w_checkbox_list1 = [
    Checkbox(
        description=f"{i}",
        indent=False,
        layout=Layout(width='36px', height='24px', margin='0 2px 0 0')
    )
    for i in range(16)
]
w_checkbox_list2 = [
    Checkbox(
        description=f"{i}",
        indent=False,
        layout=Layout(width='36px', height='24px', margin='0 2px 0 0')
    )
    for i in range(16)
]
w_new = Button(icon="plus", button_style="success",layout=widgets.Layout(width="40px"))
w_delete = Button(icon="trash", button_style="danger",layout=widgets.Layout(width="40px"))

class ArrayState:
    positions = [[0, 0]]
    cropacIdx1 = [0]
    cropacIdx2 = [0]

class State:
    array = ArrayState()
    do_update=True

state = State()

for w in (w_f, w_x, w_y):
     w.layout.width = f"{fig_width * 80 - 50}px"
    
w_idx.layout.width = f"{fig_width * 80 - 160}px"

out = Output()
def update_plot(change=None):

    if not state.do_update:
        return

    fc = w_f.value
    pattern = np.array(state.array.positions).T

    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))

    patternX, patternY = pattern
    pattern = pattern.transpose()

    k = 2 * np.pi * fc / c

    xv, yv = np.meshgrid(x, y)
    thetav = np.atan2(yv, xv)
    rv = np.sqrt(xv*xv + yv*yv)
    
    p = np.stack((k * np.cos(thetav) * np.sin(rv), k * np.sin(thetav) * np.sin(rv)), axis=-1)

    # def computeBeamPattern(micPositions):
    #     mp = micPositions
    #     if len(mp) == 0:
    #         mp = np.array([[0, 0]])
    #     return np.sum(np.array([np.exp(np.dot(p, mp[i])*1j) for i in range(len(mp))]), axis=0) / len(mp)

    # def plotMesh(ax, z):
    #     polar_pts, cart_pts, xy = plot_points
    #     interp = RegularGridInterpolator(xy, np.abs(z), bounds_error=False)
    #     z = interp(cart_pts)
    #     c = ax.pcolormesh(*polar_pts, np.abs(z), shading='auto', cmap='viridis')
    #     plt.colorbar(c, ax=ax, label='Gain (linear)')

    # idx1 = []
    # idx2 = []
    # for i in range(len(pattern)):
    #     if w_checkbox_list1[i].value:
    #         idx1.append(i)
    # for i in range(len(pattern)):
    #     if w_checkbox_list2[i].value:
    #         idx2.append(i)

    # state.array.cropacIdx1 = idx1
    # state.array.cropacIdx2 = idx2
    # subPattern1 = pattern[idx1]
    # subPattern2 = pattern[idx2]

    # z0 = computeBeamPattern(pattern)
    # z1 = computeBeamPattern(subPattern1)
    # z2 = computeBeamPattern(subPattern2)
    # z3 = np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))

    def computeZ(micPositions):
        mp = micPositions
        return np.array([np.exp(np.dot(p, mp[i])*1j) for i in range(len(mp))])

    def cropac(z1, z2):
        return np.maximum(0, np.real(2 * z1 * np.conj(z2)) / np.real(z1 * np.conj(z1) + z2 * np.conj(z2) + 1e-18))

    def computeBeamPattern(z):
        return np.sum(z, axis=0) / z.shape[0]

    def plotMesh(ax, z):
        polar_pts, cart_pts, xy = plot_points
        interp = RegularGridInterpolator(xy, np.abs(z), bounds_error=False)
        z = interp(cart_pts)
        c = ax.pcolormesh(*polar_pts, np.abs(z), shading='auto', cmap='viridis')
        plt.colorbar(c, ax=ax, label='Gain (linear)')

    idx1 = [0, 0]
    idx2 = [0, 0]

    j = 0
    for i in range(len(pattern)):
        if w_checkbox_list1[i].value and j < 2:
            idx1[j] = i
            j += 1
    j = 0
    for i in range(len(pattern)):
        if w_checkbox_list2[i].value and j < 2:
            idx2[j] = i
            j += 1

    state.array.cropacIdx1 = idx1
    state.array.cropacIdx2 = idx2

    z = computeZ(pattern)
    z0 = computeBeamPattern(z)
    z1 = cropac(z[idx1[0]], z[idx1[1]])
    z2 = cropac(z[idx2[0]], z[idx2[1]])
    z3 = z1 * z2

    with out:
        clear_output(wait=True)

        fig = plt.figure(figsize=(fig_width, fig_width * 0.55))
        ax1 = fig.add_subplot(2, 3, 1)
        ax2 = fig.add_subplot(2, 3, 2, projection='polar')
        ax3 = fig.add_subplot(2, 3, 3, projection='polar')
        ax4 = fig.add_subplot(2, 3, 4, projection='polar')
        ax5 = fig.add_subplot(2, 3, 5, projection='polar')
        ax6 = fig.add_subplot(2, 3, 6, projection='polar')
        
        rect1 = Rectangle((0,0), 0.277, 0.32, color='gray', fc = 'none',lw = 2)
        rect2 = Rectangle((0.03,0.03), 0.21, 0.26, color='gray', fc = 'none',lw = 1)
        ax1.add_patch(rect1)
        ax1.add_patch(rect2)
        ax1.scatter(patternX[idx], patternY[idx], s=100, c='C1')
        ax1.scatter(patternX, patternY, s=40, c='C0')
        ax1.scatter(patternX[idx1], patternY[idx1], s=5, c='white')
        ax1.scatter(patternX[idx2], patternY[idx2], s=5, c='black')
        ax1.axis('equal')

        plotMesh(ax2, z0)
        plotMesh(ax3, z0 * z3)
        plotMesh(ax6, z3)
        plotMesh(ax4, z1)
        plotMesh(ax5, z2)

        ax2.set_title('Delay and sum beam pattern')
        ax3.set_title('Cropac beam pattern')
        ax6.set_title('Cropac gain pattern')
        ax4.set_title('Subset beam pattern 1')
        ax5.set_title('Subset beam pattern 2')
        plt.tight_layout()
        plt.show()

def add_mic(change=None):
    state.array.positions.append([w_x.value, w_y.value])
    update_plot()

def delete_mic(change=None):
    if len(state.array.positions) > 1:
        idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
        state.array.positions.pop(idx)
        update_plot()

def set_mic_position(change=None):
    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
    state.array.positions[idx] = [w_x.value, w_y.value]
    update_plot()

def set_idx(change=None):
    idx = max(0, min(int(w_idx.value), len(state.array.positions)-1))
    x, y = state.array.positions[idx]
    state.do_update = False
    w_x.value = x
    w_y.value = y
    state.do_update = True
    update_plot()

def set_array(array):
    state.array.__dict__ = array
    state.do_update = False
    for i in range(16):
        w_checkbox_list1[i].value = False
        w_checkbox_list2[i].value = False
    for i in state.array.cropacIdx1:
        w_checkbox_list1[i].value = True
    for i in state.array.cropacIdx2:
        w_checkbox_list2[i].value = True
    x, y = state.array.positions[0]
    w_idx.value = 0
    w_x.value = x
    w_y.value = y
    state.do_update = True
    update_plot()

def get_array():
     return state.array.__dict__

w_new.on_click(add_mic)
w_delete.on_click(delete_mic)
w_x.observe(set_mic_position, names="value")
w_y.observe(set_mic_position, names="value")
w_idx.observe(set_idx, names="value")

for w in (w_f, w_idx, w_x, w_y, *w_checkbox_list1, *w_checkbox_list2):
    w.observe(update_plot, names="value")

afw = make_array_file_widgets(get_array, set_array)

display(VBox([
    afw,
    w_f,
    HBox([w_idx, w_new, w_delete]),
    w_x,
    w_y,
    HBox(w_checkbox_list1, layout=Layout(margin='0 0 0 100px')),
    HBox(w_checkbox_list2, layout=Layout(margin='0 0 0 100px')),
    out
]))


update_plot(None)